# WiSig — Module 3 v2: Confidence-aware Gated Adaptation

固定使用 Module 2 的最终输出：

- **Sid** = `z(EmbMaha) + 0.5 * z(Energy)`
- **Sdom** = `V1_mixNLL`
- **认证 Gate**：
  - `Sid < τ_id` → **Unknown**
  - `Sid ≥ τ_id 且 Sdom > τ_dom` → **Known-Drift**
  - else → **Known-Stable**

本 notebook 在你现有 Module 3 的基础上升级为 **v2**，核心变化有两点：

1. **confidence-aware target selection**
   - 先按认证 gate 选出 `Known-Drift`
   - 再用 source 模型在 stable A 上校准出的 `MSP` 阈值，筛选高置信度 target 样本
   - 若筛选后过少，则按 `MSP` 兜底保留一批

2. **更强的 adapter / 更弱的锚定**
   - `ADAPTER_SCALE` 增大
   - `ADAPT_EPOCHS` 增大
   - `LAMBDA_REPLAY_ANCHOR` 降低

## 对照组
- `NoAdapt`
- `UngatedAdapt`
- `GatedAdapt`
- `GatedConfAdapt`
- `OracleGatePseudoAdapt`
- `OracleLabelAdapt`

## 本版的关键诊断
- `pseudo_acc_rx / pseudo_acc_day / pseudo_acc_all`
- `gate_precision / gate_recall`
- `conf_precision / conf_recall`

其中最关键的是：

- **`OracleLabelAdapt`**：如果它明显高于 `OracleGatePseudoAdapt`，说明当前瓶颈主要是**伪标签噪声**
- **`GatedConfAdapt`**：如果它明显高于 `GatedAdapt`，说明 **confidence-aware 过滤是有效的**


## 新增：FALSE_DRIFT_TARGET sweep
本版本会在多个 `false_drift_target` 设定下重复评估同一 fold，同步观察：
- gate 的 precision / recall / keep ratio
- 各策略在 drift / stable / unknown 上的变化
- 更激进的 drift 触发阈值是否真的有利于适配


In [1]:
import os, json, time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, confusion_matrix

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

dataset_name = "ManySig"
dataset_path = "../ManySig.pkl/"

def load_compact_pkl_dataset(dataset_path, dataset_name):
    with open(dataset_path + dataset_name + '.pkl', 'rb') as f:
        dataset = pickle.load(f)
    return dataset

compact_dataset = load_compact_pkl_dataset(dataset_path, dataset_name)

TX_TOTAL_USE = 6
RX_TOTAL_USE = 12
DAY_TOTAL_USE = 4
KNOWN_TX_NUM = 4
SOURCE_RX_NUM = 3
TX_SPLIT_REPEATS = 5

TRAIN_DATES = ["2021_03_15"]
TEST_DATES_RX = ["2021_03_15"]
TEST_DATES_DAY = ["2021_03_01"]

Z_FROM_EQ = 1
D_FROM = "raw"
MAX_SIG_PER_TRIPLE = None

BATCH_SIZE = 128
EPOCHS = 60
LR = 1e-4
WEIGHT_DECAY = 1e-3
PATIENCE = 8
IN_PLANES = 64
DROPOUT = 0.3
D_DIM = 32

# 固定 Module 2
FUSION_LAM = 0.5
FRR_TARGET = 0.05

# ===== sweep: 不同 false_drift target =====
# 值越大 => tau_dom 越低 => 越激进，更容易把样本当成 drift
FALSE_DRIFT_TARGET_LIST = [0.01, 0.05, 0.10, 0.20]

# target stream split
ADAPT_FRAC = 0.5
MAX_ADAPT_PER_SET = 2000
MAX_EVAL_PER_SET = 3000

# ===== v2 新增：confidence-aware =====
CONF_STABLE_Q = 0.10
CONF_MIN_KEEP = 64
USE_CONF_WEIGHT = True

# ===== v2：更强的 adapter =====
ADAPTER_BOTTLENECK = 64
ADAPTER_SCALE = 0.3
ADAPT_EPOCHS = 20
ADAPT_LR = 1e-3
ADAPT_WEIGHT_DECAY = 1e-4
ADAPT_BATCH = 128

# replay
REPLAY_PER_CLASS = 200
LAMBDA_REPLAY_CE = 1.0
LAMBDA_REPLAY_ANCHOR = 1.0

ts = time.strftime("%Y%m%d_%H%M%S")
RUN_DIR = f"./results_wisig_module3_v2_false_drift_sweep/run_{ts}"
os.makedirs(RUN_DIR, exist_ok=True)

cfg = dict(
    SEED=SEED,
    TX_TOTAL_USE=TX_TOTAL_USE, RX_TOTAL_USE=RX_TOTAL_USE, DAY_TOTAL_USE=DAY_TOTAL_USE,
    KNOWN_TX_NUM=KNOWN_TX_NUM, SOURCE_RX_NUM=SOURCE_RX_NUM, TX_SPLIT_REPEATS=TX_SPLIT_REPEATS,
    TRAIN_DATES=TRAIN_DATES, TEST_DATES_RX=TEST_DATES_RX, TEST_DATES_DAY=TEST_DATES_DAY,
    Z_FROM_EQ=Z_FROM_EQ, D_FROM=D_FROM, MAX_SIG_PER_TRIPLE=MAX_SIG_PER_TRIPLE,
    BATCH_SIZE=BATCH_SIZE, EPOCHS=EPOCHS, LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, PATIENCE=PATIENCE,
    IN_PLANES=IN_PLANES, DROPOUT=DROPOUT, D_DIM=D_DIM,
    FUSION_LAM=FUSION_LAM, FRR_TARGET=FRR_TARGET, FALSE_DRIFT_TARGET_LIST=FALSE_DRIFT_TARGET_LIST,
    ADAPT_FRAC=ADAPT_FRAC, MAX_ADAPT_PER_SET=MAX_ADAPT_PER_SET, MAX_EVAL_PER_SET=MAX_EVAL_PER_SET,
    CONF_STABLE_Q=CONF_STABLE_Q, CONF_MIN_KEEP=CONF_MIN_KEEP, USE_CONF_WEIGHT=USE_CONF_WEIGHT,
    ADAPTER_BOTTLENECK=ADAPTER_BOTTLENECK, ADAPTER_SCALE=ADAPTER_SCALE,
    ADAPT_EPOCHS=ADAPT_EPOCHS, ADAPT_LR=ADAPT_LR, ADAPT_WEIGHT_DECAY=ADAPT_WEIGHT_DECAY, ADAPT_BATCH=ADAPT_BATCH,
    REPLAY_PER_CLASS=REPLAY_PER_CLASS, LAMBDA_REPLAY_CE=LAMBDA_REPLAY_CE, LAMBDA_REPLAY_ANCHOR=LAMBDA_REPLAY_ANCHOR,
)
with open(os.path.join(RUN_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2)

print("RUN_DIR:", RUN_DIR)

Device: cuda
RUN_DIR: ./results_wisig_module3_v2_false_drift_sweep/run_20260307_114235


In [2]:
def get_idx(lst, val):
    return lst.index(val)

def subsample(sigs, max_sig):
    if max_sig is None:
        return sigs
    return sigs[:min(len(sigs), max_sig)]

def get_signals(compact_dataset, tx, rx, date, equalized):
    tx_i = get_idx(compact_dataset["tx_list"], tx)
    rx_i = get_idx(compact_dataset["rx_list"], rx)
    date_i = get_idx(compact_dataset["capture_date_list"], date)
    eq_i = get_idx(compact_dataset["equalized_list"], equalized)
    return np.array(compact_dataset["data"][tx_i][rx_i][date_i][eq_i], dtype=np.float32)

def iq_to_complex(x):
    return (x[...,0] + 1j*x[...,1]).astype(np.complex64)

def domain_feat_256_complex(x256_c, d_dim=32):
    x = x256_c / (np.sqrt(np.mean(np.abs(x256_c)**2)) + 1e-12)
    X = np.fft.fft(x, n=256)
    mag = np.abs(X) + 1e-12
    logm = np.log(mag)
    D = np.fft.rfft(logm, n=512)
    return np.real(D[:d_dim]).astype(np.float32)

def compute_domain_feats_batch(X, d_dim=32):
    Xc = iq_to_complex(X)
    return np.stack([domain_feat_256_complex(Xc[i], d_dim=d_dim) for i in range(Xc.shape[0])], axis=0).astype(np.float32)

tx_list = compact_dataset["tx_list"]
rx_list = compact_dataset["rx_list"]
date_list = compact_dataset["capture_date_list"]
TX_USE = tx_list[:TX_TOTAL_USE]
RX_USE = rx_list[:RX_TOTAL_USE]
DATE_USE = date_list[:DAY_TOTAL_USE]
rng_rx = np.random.RandomState(SEED)
SOURCE_RXS = list(rng_rx.choice(RX_USE, size=SOURCE_RX_NUM, replace=False))
SOURCE_RXS.sort()
DRIFT_RX = [r for r in RX_USE if r not in SOURCE_RXS]
print("SOURCE_RXS:", SOURCE_RXS, "DRIFT_RX:", DRIFT_RX)

SOURCE_RXS: [np.str_('1-1'), np.str_('7-14'), np.str_('7-7')] DRIFT_RX: ['1-19', '14-7', '18-2', '19-2', '2-1', '2-19', '20-1', '3-19', '8-8']


In [3]:
class BasicBlock1D(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, downsample=None, dropout=0.0):
        super().__init__()
        self.conv1 = nn.Conv1d(in_planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(planes, planes, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(planes)
        self.downsample = downsample
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.dropout(out)
        out = self.bn2(self.conv2(out))
        if self.downsample is not None:
            identity = self.downsample(x)
        return self.relu(out + identity)

class ResNet18_1D(nn.Module):
    def __init__(self, num_classes, in_planes=64, dropout=0.0):
        super().__init__()
        self.in_planes = in_planes
        self.conv1 = nn.Conv1d(2, in_planes, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(in_planes)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool1d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, stride=1, dropout=dropout)
        self.layer2 = self._make_layer(128, 2, stride=2, dropout=dropout)
        self.layer3 = self._make_layer(256, 2, stride=2, dropout=dropout)
        self.layer4 = self._make_layer(512, 2, stride=2, dropout=dropout)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(512, num_classes)
    def _make_layer(self, planes, blocks, stride, dropout):
        downsample = None
        if stride != 1 or self.in_planes != planes:
            downsample = nn.Sequential(
                nn.Conv1d(self.in_planes, planes, 1, stride=stride, bias=False),
                nn.BatchNorm1d(planes)
            )
        layers = [BasicBlock1D(self.in_planes, planes, stride, downsample, dropout)]
        self.in_planes = planes
        for _ in range(1, blocks):
            layers.append(BasicBlock1D(self.in_planes, planes, dropout=dropout))
        return nn.Sequential(*layers)
    def forward(self, x, return_embed=False):
        x = x.permute(0, 2, 1)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x).squeeze(-1)
        logits = self.fc(x)
        if return_embed:
            return logits, x
        return logits

In [4]:
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, i):
        return self.X[i], self.y[i]

def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    crit = nn.CrossEntropyLoss()
    total_loss, total_correct, total = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        if train:
            optimizer.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        if train:
            loss.backward()
            optimizer.step()
        total_loss += float(loss.item()) * yb.size(0)
        total_correct += int((logits.argmax(1) == yb).sum().item())
        total += int(yb.size(0))
    return total_loss/max(1,total), total_correct/max(1,total)

def infer_logits_embed(model, X_np, batch=512):
    model.eval()
    ds = ArrayDataset(X_np, np.zeros((len(X_np),), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    logits_all, Z_all = [], []
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits, emb = model(xb, return_embed=True)
            logits_all.append(logits.cpu().numpy().astype(np.float32))
            Z_all.append(emb.cpu().numpy().astype(np.float32))
    return np.concatenate(logits_all,0), np.concatenate(Z_all,0)

def closed_set_acc_from_logits(logits, y_true):
    return float((np.argmax(logits,1) == y_true).mean())

def roc_auc(y_true, score):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(auc(fpr, tpr))

def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / (ez.sum(axis=1, keepdims=True) + 1e-12)

def msp_from_logits(logits):
    return softmax_np(logits).max(axis=1).astype(np.float32)

def pseudo_acc(logits, y_true):
    pred = np.argmax(logits, axis=1)
    mask = y_true >= 0
    return float((pred[mask] == y_true[mask]).mean()) if np.any(mask) else float("nan")

In [5]:
def fit_emb_maha_diag(Z_train, y_train, K):
    mu = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    var = np.zeros((K, Z_train.shape[1]), dtype=np.float32)
    for k in range(K):
        Zk = Z_train[y_train==k]
        mu[k] = Zk.mean(axis=0)
        var[k] = Zk.var(axis=0) + 1e-3
    return mu, var

def sid_embmaha(Z, mu_z, var_z):
    dif = Z[:,None,:] - mu_z[None,:,:]
    dist = np.sum((dif*dif)/(var_z[None,:,:] + 1e-6), axis=2)
    return (-np.min(dist, axis=1)).astype(np.float32)

def sid_energy(logits):
    m = logits.max(axis=1, keepdims=True)
    return (m + np.log(np.exp(logits-m).sum(axis=1, keepdims=True)+1e-12)).squeeze(1).astype(np.float32)

def zscore_fixed(x, ref):
    mu = float(np.mean(ref))
    std = float(np.std(ref))
    if std < 1e-12:
        std = 1.0
    return ((x - mu) / std).astype(np.float32)

def sid_fusion_fixed(Z, logits, mu_z, var_z, ref_sid_emb_A, ref_sid_en_A, lam=0.5):
    return (
        zscore_fixed(sid_embmaha(Z, mu_z, var_z), ref_sid_emb_A)
        + lam * zscore_fixed(sid_energy(logits), ref_sid_en_A)
    ).astype(np.float32)

def mahalanobis_batch(D, mu, Sinv):
    X = D - mu.reshape(1,-1)
    return np.einsum("nd,dd,nd->n", X, Sinv, X).astype(np.float32)

def fit_gaussian(D, reg=1e-3):
    mu = D.mean(axis=0).astype(np.float32)
    C = np.cov(D.T, bias=False).astype(np.float32)
    C = C + reg*np.eye(C.shape[0], dtype=np.float32)
    Sinv = np.linalg.inv(C).astype(np.float32)
    sign, logdet = np.linalg.slogdet(C)
    if sign <= 0:
        logdet = float(np.log(np.maximum(np.linalg.det(C), 1e-12)))
    return mu, Sinv, float(logdet)

def logpdf_gaussian(D, mu, Sinv, logdet):
    d = D.shape[1]
    maha = mahalanobis_batch(D, mu, Sinv)
    return (-0.5*(maha + logdet + d*np.log(2*np.pi))).astype(np.float32)

def fit_device_rx_models(D_train, y_train, rx_train, K, source_rx_ids, reg=1e-3, min_n=20):
    models = {}
    fallback = {}
    for k in range(K):
        fallback[k] = fit_gaussian(D_train[y_train==k], reg=reg)
        for rxid in source_rx_ids:
            idx = np.where((y_train==k) & (rx_train==rxid))[0]
            if idx.size >= min_n:
                models[(k,rxid)] = fit_gaussian(D_train[idx], reg=reg)
    return models, fallback

def sdom_V1_mixNLL(D_eval, k_hat, models_kr, fallback_k, source_rx_ids):
    N = D_eval.shape[0]
    out = np.zeros((N,), dtype=np.float32)
    R = len(source_rx_ids)
    for i in range(N):
        k = int(k_hat[i])
        d = D_eval[i:i+1]
        logps = []
        for rxid in source_rx_ids:
            mu,Sinv,logdet = models_kr.get((k,rxid), fallback_k[k])
            logps.append(float(logpdf_gaussian(d, mu, Sinv, logdet)[0]))
        logps = np.array(logps, dtype=np.float32)
        m = logps.max()
        loglik = m + np.log(np.exp(logps-m).sum()+1e-12) - np.log(R)
        out[i] = float(-loglik)
    return out

def calibrate_module2_thresholds(Sid_A, Sdom_A, frr_target=0.05, false_drift_target=0.05):
    tau_id = float(np.quantile(Sid_A, frr_target))
    tau_dom = float(np.quantile(Sdom_A, 1.0 - false_drift_target))
    return tau_id, tau_dom

def gate_predict(Sid, Sdom, tau_id, tau_dom):
    pred = np.zeros_like(Sid, dtype=np.int64)
    pred[Sid < tau_id] = 2
    pred[(Sid >= tau_id) & (Sdom > tau_dom)] = 1
    return pred

def compute_module2_scores_from_cached(Z, logits, D, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, models_kr, fallback_k, source_rx_ids, tau_id=None, tau_dom=None):
    k_hat = np.argmax(logits, axis=1).astype(np.int64)
    Sid = sid_fusion_fixed(Z, logits, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A, lam=FUSION_LAM)
    Sdom = sdom_V1_mixNLL(D, k_hat, models_kr, fallback_k, source_rx_ids)
    out = dict(Sid=Sid, Sdom=Sdom, k_hat=k_hat)
    if tau_id is not None and tau_dom is not None:
        out["gate_pred"] = gate_predict(Sid, Sdom, tau_id, tau_dom)
    return out

In [6]:
def build_source_train(compact_dataset, KNOWN_TX):
    X_list, y_list, D_list, rx_id_list = [], [], [], []
    for tx in KNOWN_TX:
        for rx in SOURCE_RXS:
            for day in TRAIN_DATES:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
                rx_id_list.append(np.full((n,), RX_USE.index(rx), dtype=np.int64))
    return (
        np.concatenate(X_list,0).astype(np.float32),
        np.concatenate(y_list,0).astype(np.int64),
        np.concatenate(D_list,0).astype(np.float32),
        np.concatenate(rx_id_list,0).astype(np.int64),
    )

def build_eval_set(compact_dataset, KNOWN_TX, txs, rxs, days):
    X_list, y_list, D_list = [], [], []
    for tx in txs:
        for rx in rxs:
            for day in days:
                Xz = subsample(get_signals(compact_dataset, tx, rx, day, Z_FROM_EQ), MAX_SIG_PER_TRIPLE)
                Xd = subsample(get_signals(compact_dataset, tx, rx, day, 0 if D_FROM=="raw" else 1), MAX_SIG_PER_TRIPLE)
                n = min(len(Xz), len(Xd))
                Xz = Xz[:n]; Xd = Xd[:n]
                y = np.full((n,), KNOWN_TX.index(tx), dtype=np.int64) if tx in KNOWN_TX else np.full((n,), -1, dtype=np.int64)
                Df = compute_domain_feats_batch(Xd, d_dim=D_DIM)
                X_list.append(Xz); y_list.append(y); D_list.append(Df)
    return (
        np.concatenate(X_list,0).astype(np.float32),
        np.concatenate(y_list,0).astype(np.int64),
        np.concatenate(D_list,0).astype(np.float32),
    )

def split_adapt_eval(X, y, D, frac=0.5, max_adapt=None, max_eval=None, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.permutation(len(X))
    cut = int(len(idx) * frac)
    idx_ad = idx[:cut]
    idx_ev = idx[cut:]
    if max_adapt is not None and len(idx_ad) > max_adapt:
        idx_ad = idx_ad[:max_adapt]
    if max_eval is not None and len(idx_ev) > max_eval:
        idx_ev = idx_ev[:max_eval]
    return X[idx_ad], y[idx_ad], D[idx_ad], X[idx_ev], y[idx_ev], D[idx_ev]

def concat_sets(items):
    X = np.concatenate([it[0] for it in items], axis=0)
    y = np.concatenate([it[1] for it in items], axis=0)
    D = np.concatenate([it[2] for it in items], axis=0)
    return X, y, D

In [7]:
class EmbDatasetWeighted(Dataset):
    def __init__(self, Z, y, w=None):
        self.Z = torch.tensor(Z, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        if w is None:
            w = np.ones((len(Z),), dtype=np.float32)
        self.w = torch.tensor(w, dtype=torch.float32)
    def __len__(self):
        return self.Z.shape[0]
    def __getitem__(self, i):
        return self.Z[i], self.y[i], self.w[i]

class ResidualAdapter(nn.Module):
    def __init__(self, dim=512, bottleneck=64, scale=0.3):
        super().__init__()
        self.fc1 = nn.Linear(dim, bottleneck)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(bottleneck, dim)
        self.scale = scale
        nn.init.zeros_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)
    def forward(self, z):
        return z + self.scale * self.fc2(self.relu(self.fc1(z)))

def apply_adapter_and_fc(adapter, fc_layer, Z_np, batch=512):
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    if adapter is None:
        with torch.no_grad():
            z = torch.tensor(Z_np, dtype=torch.float32)
            logits = fc(z).numpy().astype(np.float32)
        return Z_np.astype(np.float32), logits

    fc = fc.to(device)
    for p in fc.parameters():
        p.requires_grad = False
    adapter.eval()

    ds = EmbDatasetWeighted(Z_np, np.zeros((len(Z_np),), dtype=np.int64))
    dl = DataLoader(ds, batch_size=batch, shuffle=False)
    Z2_all, logits_all = [], []
    with torch.no_grad():
        for zb,_,_ in dl:
            zb = zb.to(device)
            z2 = adapter(zb)
            logits = fc(z2)
            Z2_all.append(z2.cpu().numpy().astype(np.float32))
            logits_all.append(logits.cpu().numpy().astype(np.float32))
    return np.concatenate(Z2_all,0), np.concatenate(logits_all,0)

def make_replay_subset(Z_src, y_src, per_class=200, seed=0):
    rng = np.random.RandomState(seed)
    keep = []
    for k in np.unique(y_src):
        idx = np.where(y_src == k)[0]
        rng.shuffle(idx)
        keep.append(idx[:min(len(idx), per_class)])
    keep = np.concatenate(keep, axis=0)
    return Z_src[keep], y_src[keep]

def normalize_conf_weights(conf):
    conf = np.asarray(conf, dtype=np.float32)
    if len(conf) == 0:
        return conf
    cmin, cmax = float(conf.min()), float(conf.max())
    if cmax - cmin < 1e-12:
        return np.ones_like(conf, dtype=np.float32)
    return (0.5 + 0.5 * (conf - cmin) / (cmax - cmin)).astype(np.float32)

def adapt_on_embeddings(fc_layer, Z_target, y_target, Z_replay, y_replay,
                        target_weights=None,
                        bottleneck=64, scale=0.3, epochs=20, lr=1e-3, wd=1e-4, batch=128,
                        lambda_replay_ce=1.0, lambda_replay_anchor=1.0):
    if len(Z_target) == 0:
        return None

    adapter = ResidualAdapter(dim=Z_target.shape[1], bottleneck=bottleneck, scale=scale).to(device)
    fc = nn.Linear(fc_layer.in_features, fc_layer.out_features, bias=True).to(device)
    fc.load_state_dict(fc_layer.state_dict())
    fc.eval()
    for p in fc.parameters():
        p.requires_grad = False

    ds_t = EmbDatasetWeighted(Z_target, y_target, target_weights)
    ds_s = EmbDatasetWeighted(Z_replay, y_replay, None)
    dl_t = DataLoader(ds_t, batch_size=batch, shuffle=True, drop_last=False)
    dl_s = DataLoader(ds_s, batch_size=batch, shuffle=True, drop_last=True)

    opt = optim.Adam(adapter.parameters(), lr=lr, weight_decay=wd)
    ce_none = nn.CrossEntropyLoss(reduction="none")
    ce = nn.CrossEntropyLoss()
    mse = nn.MSELoss()

    for ep in range(epochs):
        adapter.train()
        it_s = iter(dl_s)
        for zt, yt, wt in dl_t:
            try:
                zs, ys, _ = next(it_s)
            except StopIteration:
                it_s = iter(dl_s)
                zs, ys, _ = next(it_s)

            zt, yt, wt = zt.to(device), yt.to(device), wt.to(device)
            zs, ys = zs.to(device), ys.to(device)

            opt.zero_grad()
            zt2 = adapter(zt)
            zs2 = adapter(zs)
            logits_t = fc(zt2)
            logits_s = fc(zs2)

            loss_t = (ce_none(logits_t, yt) * wt).mean()
            loss_s = ce(logits_s, ys)
            loss_anchor = mse(zs2, zs)
            loss = loss_t + lambda_replay_ce * loss_s + lambda_replay_anchor * loss_anchor
            loss.backward()
            opt.step()

    return adapter

def select_confident_from_gated(gate_pred, msp_target, tau_conf, min_keep=64):
    idx_gate = np.where(gate_pred == 1)[0]
    if len(idx_gate) == 0:
        return idx_gate

    idx_conf = idx_gate[msp_target[idx_gate] >= tau_conf]
    if len(idx_conf) >= min_keep:
        return idx_conf

    order = np.argsort(-msp_target[idx_gate])
    k = min(len(idx_gate), min_keep)
    return idx_gate[order[:k]]

def plot_bar_compare(metric_dict, out_png, title):
    names = list(metric_dict.keys())
    vals = [metric_dict[k] for k in names]
    plt.figure(figsize=(8,4))
    plt.bar(range(len(names)), vals)
    plt.xticks(range(len(names)), names, rotation=20)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_png, dpi=160)
    plt.close()

In [8]:
def evaluate_under_thresholds(adapter, model_fc, Z_A, D_A, y_A,
                              Z_B, D_B, y_B,
                              Z_C, D_C, y_C,
                              Z_U, D_U,
                              mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A,
                              models_kr, fallback_k, source_rx_ids,
                              tau_id, tau_dom):
    Z_A2, logits_A2 = apply_adapter_and_fc(adapter, model_fc, Z_A, batch=512)
    Z_B2, logits_B2 = apply_adapter_and_fc(adapter, model_fc, Z_B, batch=512)
    Z_C2, logits_C2 = apply_adapter_and_fc(adapter, model_fc, Z_C, batch=512)
    Z_U2, logits_U2 = apply_adapter_and_fc(adapter, model_fc, Z_U, batch=512)

    sc_A2 = compute_module2_scores_from_cached(
        Z_A2, logits_A2, D_A, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A,
        models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom
    )
    sc_B2 = compute_module2_scores_from_cached(
        Z_B2, logits_B2, D_B, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A,
        models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom
    )
    sc_C2 = compute_module2_scores_from_cached(
        Z_C2, logits_C2, D_C, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A,
        models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom
    )
    sc_U2 = compute_module2_scores_from_cached(
        Z_U2, logits_U2, D_U, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A,
        models_kr, fallback_k, source_rx_ids, tau_id=tau_id, tau_dom=tau_dom
    )

    stable_acc = closed_set_acc_from_logits(logits_A2, y_A)
    drift_acc_rx = closed_set_acc_from_logits(logits_B2, y_B)
    drift_acc_day = closed_set_acc_from_logits(logits_C2, y_C)
    drift_acc_all = float(
        (np.sum(np.argmax(logits_B2,1) == y_B) + np.sum(np.argmax(logits_C2,1) == y_C)) /
        (len(y_B) + len(y_C))
    )

    FRR_stable = float(1.0 - (sc_A2["gate_pred"] == 0).mean())
    FAR_unknown_accept = float((sc_U2["gate_pred"] == 0).mean())
    miss_drift_rx = float((sc_B2["gate_pred"] == 0).mean())
    miss_drift_day = float((sc_C2["gate_pred"] == 0).mean())
    miss_drift_all = float(
        (np.sum(sc_B2["gate_pred"] == 0) + np.sum(sc_C2["gate_pred"] == 0)) /
        (len(sc_B2["gate_pred"]) + len(sc_C2["gate_pred"]))
    )
    auc_unknown_eval = roc_auc(
        np.concatenate([np.zeros_like(sc_A2["Sid"], dtype=np.int64), np.ones_like(sc_U2["Sid"], dtype=np.int64)]),
        np.concatenate([-sc_A2["Sid"], -sc_U2["Sid"]])
    )

    return dict(
        stable_acc=stable_acc,
        drift_acc_rx=drift_acc_rx,
        drift_acc_day=drift_acc_day,
        drift_acc_all=drift_acc_all,
        FRR_stable=FRR_stable,
        FAR_unknown_accept=FAR_unknown_accept,
        miss_drift_rx=miss_drift_rx,
        miss_drift_day=miss_drift_day,
        miss_drift_all=miss_drift_all,
        auc_unknown_eval=auc_unknown_eval,
    )

def run_one_split(split_id, KNOWN_TX, UNKNOWN_TX):
    split_dir = os.path.join(RUN_DIR, f"txsplit_{split_id}")
    os.makedirs(split_dir, exist_ok=True)
    with open(os.path.join(split_dir, "tx_split.json"), "w", encoding="utf-8") as f:
        json.dump({"KNOWN_TX": KNOWN_TX, "UNKNOWN_TX": UNKNOWN_TX, "SOURCE_RXS": SOURCE_RXS, "DRIFT_RX": DRIFT_RX}, f, indent=2)

    X_src, y_src, D_src, rx_src = build_source_train(compact_dataset, KNOWN_TX)
    K = len(KNOWN_TX)

    X_B, y_B, D_B = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_C, y_C, D_C = build_eval_set(compact_dataset, KNOWN_TX, KNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)
    X_D, y_D, D_D = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_RX)
    X_E, y_E, D_E = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, DRIFT_RX, TEST_DATES_RX)
    X_F, y_F, D_F = build_eval_set(compact_dataset, KNOWN_TX, UNKNOWN_TX, SOURCE_RXS, TEST_DATES_DAY)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1000*split_id)
    rows = []

    for fold, (idx_tr, idx_te) in enumerate(skf.split(X_src, y_src), start=1):
        fold_dir = os.path.join(split_dir, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        rng = np.random.RandomState(SEED + 1000*split_id + fold)
        perm = rng.permutation(idx_tr)
        n_val = max(1, int(0.1 * len(perm)))
        idx_val = perm[:n_val]
        idx_train = perm[n_val:]

        # source training
        train_loader = DataLoader(ArrayDataset(X_src[idx_train], y_src[idx_train]), batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(ArrayDataset(X_src[idx_val],   y_src[idx_val]),   batch_size=BATCH_SIZE, shuffle=False)

        model = ResNet18_1D(num_classes=K, in_planes=IN_PLANES, dropout=DROPOUT).to(device)
        opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

        best_val = -1.0
        best_state = None
        patience = 0
        for ep in range(1, EPOCHS + 1):
            _ = run_epoch(model, train_loader, optimizer=opt)
            _, va_acc = run_epoch(model, val_loader, optimizer=None)
            if va_acc > best_val + 1e-4:
                best_val = va_acc
                best_state = {k:v.cpu().clone() for k,v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1
                if patience >= PATIENCE:
                    break

        model.load_state_dict(best_state)
        torch.save(best_state, os.path.join(fold_dir, "best_source_model.pth"))

        # source caches
        logits_tr, Z_tr = infer_logits_embed(model, X_src[idx_train], batch=512)
        logits_A, Z_A = infer_logits_embed(model, X_src[idx_te], batch=512)
        D_A = D_src[idx_te]

        mu_z_src, var_z_src = fit_emb_maha_diag(Z_tr, y_src[idx_train], K)
        ref_sid_emb_A = sid_embmaha(Z_A, mu_z_src, var_z_src)
        ref_sid_en_A  = sid_energy(logits_A)
        msp_A = msp_from_logits(logits_A)

        source_rx_ids = sorted({RX_USE.index(r) for r in SOURCE_RXS})
        models_kr, fallback_k = fit_device_rx_models(D_src[idx_train], y_src[idx_train], rx_src[idx_train], K, source_rx_ids, reg=1e-3, min_n=20)

        # pseudo-label diagnostics on true drift pools (independent of false_drift_target)
        logits_B_ev0, Z_B_ev0 = infer_logits_embed(model, X_B, batch=512)
        logits_C_ev0, Z_C_ev0 = infer_logits_embed(model, X_C, batch=512)
        pseudo_acc_rx = pseudo_acc(logits_B_ev0, y_B)
        pseudo_acc_day = pseudo_acc(logits_C_ev0, y_C)
        pseudo_acc_all = float(
            (np.sum(np.argmax(logits_B_ev0,1) == y_B) + np.sum(np.argmax(logits_C_ev0,1) == y_C)) /
            (len(y_B) + len(y_C))
        )
        tau_conf = float(np.quantile(msp_A, CONF_STABLE_Q))

        # split pools once; reused across all false_drift settings
        B_sp = split_adapt_eval(X_B, y_B, D_B, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=10000+fold)
        C_sp = split_adapt_eval(X_C, y_C, D_C, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=11000+fold)
        D_sp = split_adapt_eval(X_D, y_D, D_D, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=12000+fold)
        E_sp = split_adapt_eval(X_E, y_E, D_E, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=13000+fold)
        F_sp = split_adapt_eval(X_F, y_F, D_F, frac=ADAPT_FRAC, max_adapt=MAX_ADAPT_PER_SET, max_eval=MAX_EVAL_PER_SET, seed=14000+fold)

        X_adapt, y_adapt, D_adapt = concat_sets([
            (B_sp[0], B_sp[1], B_sp[2]),
            (C_sp[0], C_sp[1], C_sp[2]),
            (D_sp[0], D_sp[1], D_sp[2]),
            (E_sp[0], E_sp[1], E_sp[2]),
            (F_sp[0], F_sp[1], F_sp[2]),
        ])
        adapt_gt_group = np.concatenate([
            np.ones(len(B_sp[0]), dtype=np.int64),
            np.ones(len(C_sp[0]), dtype=np.int64),
            np.full(len(D_sp[0]), 2, dtype=np.int64),
            np.full(len(E_sp[0]), 2, dtype=np.int64),
            np.full(len(F_sp[0]), 2, dtype=np.int64),
        ])

        X_B_ev, y_B_ev, D_B_ev = B_sp[3], B_sp[4], B_sp[5]
        X_C_ev, y_C_ev, D_C_ev = C_sp[3], C_sp[4], C_sp[5]
        X_U_ev, y_U_ev, D_U_ev = concat_sets([
            (D_sp[3], D_sp[4], D_sp[5]),
            (E_sp[3], E_sp[4], E_sp[5]),
            (F_sp[3], F_sp[4], F_sp[5]),
        ])

        logits_adapt, Z_adapt = infer_logits_embed(model, X_adapt, batch=512)
        logits_B_ev, Z_B_ev = infer_logits_embed(model, X_B_ev, batch=512)
        logits_C_ev, Z_C_ev = infer_logits_embed(model, X_C_ev, batch=512)
        logits_U_ev, Z_U_ev = infer_logits_embed(model, X_U_ev, batch=512)

        yhat_adapt = np.argmax(logits_adapt, axis=1).astype(np.int64)
        msp_adapt = msp_from_logits(logits_adapt)
        Z_replay, y_replay = make_replay_subset(Z_tr, y_src[idx_train], per_class=REPLAY_PER_CLASS, seed=SEED+fold)
        idx_all = np.arange(len(X_adapt))
        idx_oracle = np.where(adapt_gt_group == 1)[0]

        # adapters independent of false_drift_target
        common_adapters = {
            "NoAdapt": None,
            "UngatedAdapt": adapt_on_embeddings(
                model.fc, Z_adapt[idx_all], yhat_adapt[idx_all], Z_replay, y_replay,
                target_weights=np.ones((len(idx_all),), dtype=np.float32),
                bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                epochs=ADAPT_EPOCHS, lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
            ),
            "OracleGatePseudoAdapt": adapt_on_embeddings(
                model.fc, Z_adapt[idx_oracle], yhat_adapt[idx_oracle], Z_replay, y_replay,
                target_weights=np.ones((len(idx_oracle),), dtype=np.float32),
                bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                epochs=ADAPT_EPOCHS, lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
            ),
            "OracleLabelAdapt": adapt_on_embeddings(
                model.fc, Z_adapt[idx_oracle], y_adapt[idx_oracle], Z_replay, y_replay,
                target_weights=np.ones((len(idx_oracle),), dtype=np.float32),
                bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                epochs=ADAPT_EPOCHS, lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
            ),
        }

        # base stable scores on A reused for tau_id/tau_dom calibration
        sc_A_base = compute_module2_scores_from_cached(
            Z_A, logits_A, D_A, mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A,
            models_kr, fallback_k, source_rx_ids
        )

        for false_drift_target in FALSE_DRIFT_TARGET_LIST:
            tau_id, tau_dom = calibrate_module2_thresholds(
                sc_A_base["Sid"], sc_A_base["Sdom"],
                FRR_TARGET, false_drift_target
            )

            # gate on adaptation stream under current false_drift_target
            sc_adapt = compute_module2_scores_from_cached(
                Z_adapt, logits_adapt, D_adapt,
                mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A,
                models_kr, fallback_k, source_rx_ids,
                tau_id=tau_id, tau_dom=tau_dom
            )
            gate_pred_adapt = sc_adapt["gate_pred"]
            idx_gate = np.where(gate_pred_adapt == 1)[0]

            idx_conf = select_confident_from_gated(gate_pred_adapt, msp_adapt, tau_conf, min_keep=CONF_MIN_KEEP)
            w_conf = normalize_conf_weights(msp_adapt[idx_conf]) if USE_CONF_WEIGHT else np.ones((len(idx_conf),), dtype=np.float32)

            gate_precision = float((adapt_gt_group[idx_gate] == 1).mean()) if len(idx_gate) > 0 else float("nan")
            gate_recall = float((gate_pred_adapt[adapt_gt_group == 1] == 1).mean()) if np.any(adapt_gt_group == 1) else float("nan")
            gate_keep_ratio = float(len(idx_gate) / len(idx_all))

            conf_precision = float((adapt_gt_group[idx_conf] == 1).mean()) if len(idx_conf) > 0 else float("nan")
            conf_recall = float((adapt_gt_group[idx_conf] == 1).sum() / max(1, (adapt_gt_group == 1).sum()))
            conf_keep_ratio = float(len(idx_conf) / len(idx_all))
            conf_keep_given_gate = float(len(idx_conf) / max(1, len(idx_gate)))

            # adapters depending on false_drift_target
            gated_adapters = {
                "GatedAdapt": adapt_on_embeddings(
                    model.fc, Z_adapt[idx_gate], yhat_adapt[idx_gate], Z_replay, y_replay,
                    target_weights=np.ones((len(idx_gate),), dtype=np.float32),
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    epochs=ADAPT_EPOCHS, lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
                ),
                "GatedConfAdapt": adapt_on_embeddings(
                    model.fc, Z_adapt[idx_conf], yhat_adapt[idx_conf], Z_replay, y_replay,
                    target_weights=w_conf,
                    bottleneck=ADAPTER_BOTTLENECK, scale=ADAPTER_SCALE,
                    epochs=ADAPT_EPOCHS, lr=ADAPT_LR, wd=ADAPT_WEIGHT_DECAY, batch=ADAPT_BATCH,
                    lambda_replay_ce=LAMBDA_REPLAY_CE, lambda_replay_anchor=LAMBDA_REPLAY_ANCHOR
                ),
            }

            adapters = {}
            adapters.update(common_adapters)
            adapters.update(gated_adapters)

            fold_metrics = dict(
                split=split_id, fold=fold, false_drift_target=float(false_drift_target),
                tau_id=tau_id, tau_dom=tau_dom, tau_conf=tau_conf,
                pseudo_acc_rx=pseudo_acc_rx, pseudo_acc_day=pseudo_acc_day, pseudo_acc_all=pseudo_acc_all,
                gate_precision=gate_precision, gate_recall=gate_recall, gate_keep_ratio=gate_keep_ratio,
                conf_precision=conf_precision, conf_recall=conf_recall, conf_keep_ratio=conf_keep_ratio,
                conf_keep_given_gate=conf_keep_given_gate,
                adapt_stream_size=int(len(idx_all)), gate_size=int(len(idx_gate)), conf_size=int(len(idx_conf)), oracle_size=int(len(idx_oracle)),
            )

            for name, adapter in adapters.items():
                fold_metrics[name] = evaluate_under_thresholds(
                    adapter, model.fc,
                    Z_A, D_A, y_src[idx_te],
                    Z_B_ev, D_B_ev, y_B_ev,
                    Z_C_ev, D_C_ev, y_C_ev,
                    Z_U_ev, D_U_ev,
                    mu_z_src, var_z_src, ref_sid_emb_A, ref_sid_en_A,
                    models_kr, fallback_k, source_rx_ids,
                    tau_id, tau_dom
                )

            fd_tag = f"fd{int(round(false_drift_target * 100)):02d}"
            plot_bar_compare(
                {k: fold_metrics[k]["drift_acc_all"] for k in ["NoAdapt","UngatedAdapt","GatedAdapt","GatedConfAdapt","OracleGatePseudoAdapt","OracleLabelAdapt"]},
                os.path.join(fold_dir, f"drift_acc_all_compare_{fd_tag}.png"),
                f"Drift accuracy compare ({fd_tag})"
            )
            plot_bar_compare(
                {k: fold_metrics[k]["FAR_unknown_accept"] for k in ["NoAdapt","UngatedAdapt","GatedAdapt","GatedConfAdapt","OracleGatePseudoAdapt","OracleLabelAdapt"]},
                os.path.join(fold_dir, f"far_unknown_compare_{fd_tag}.png"),
                f"Unknown FAR compare ({fd_tag})"
            )

            with open(os.path.join(fold_dir, f"metrics_module3_v2_{fd_tag}.json"), "w", encoding="utf-8") as f:
                json.dump(fold_metrics, f, indent=2)

            rows.append(fold_metrics)
            print(
                f"[split {split_id} fold {fold} fd={false_drift_target:.2f}] "
                f"gateP={gate_precision:.3f} gateR={gate_recall:.3f} keep={gate_keep_ratio:.3f} | "
                f"NoAdapt driftAll={fold_metrics['NoAdapt']['drift_acc_all']:.3f} FARunk={fold_metrics['NoAdapt']['FAR_unknown_accept']:.3f} | "
                f"Gated driftAll={fold_metrics['GatedAdapt']['drift_acc_all']:.3f} FARunk={fold_metrics['GatedAdapt']['FAR_unknown_accept']:.3f}"
            )

    with open(os.path.join(split_dir, "metrics_all_folds.json"), "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)
    return rows

all_rows = []
for split_id in range(1, TX_SPLIT_REPEATS + 1):
    rng_tx = np.random.RandomState(SEED + 777*split_id)
    tx_perm = list(rng_tx.permutation(TX_USE))
    KNOWN_TX = tx_perm[:KNOWN_TX_NUM]
    UNKNOWN_TX = tx_perm[KNOWN_TX_NUM:]
    print("\n=== TX split", split_id, "===")
    print("KNOWN_TX:", KNOWN_TX)
    print("UNKNOWN_TX:", UNKNOWN_TX)
    all_rows.extend(run_one_split(split_id, KNOWN_TX, UNKNOWN_TX))

with open(os.path.join(RUN_DIR, "metrics_all_splits_folds.json"), "w", encoding="utf-8") as f:
    json.dump(all_rows, f, indent=2)
print("\nSaved all metrics to:", RUN_DIR)


=== TX split 1 ===
KNOWN_TX: [np.str_('20-15'), np.str_('20-19'), np.str_('14-10'), np.str_('8-20')]
UNKNOWN_TX: [np.str_('6-15'), np.str_('14-7')]
[split 1 fold 1 fd=0.01] gateP=0.246 gateR=0.051 keep=0.082 | NoAdapt driftAll=0.625 FARunk=0.592 | Gated driftAll=0.627 FARunk=0.587
[split 1 fold 1 fd=0.05] gateP=0.258 gateR=0.133 keep=0.206 | NoAdapt driftAll=0.625 FARunk=0.440 | Gated driftAll=0.627 FARunk=0.463
[split 1 fold 1 fd=0.10] gateP=0.238 gateR=0.200 keep=0.336 | NoAdapt driftAll=0.625 FARunk=0.268 | Gated driftAll=0.628 FARunk=0.283
[split 1 fold 1 fd=0.20] gateP=0.223 gateR=0.274 keep=0.492 | NoAdapt driftAll=0.625 FARunk=0.056 | Gated driftAll=0.628 FARunk=0.056
[split 1 fold 2 fd=0.01] gateP=0.240 gateR=0.051 keep=0.085 | NoAdapt driftAll=0.625 FARunk=0.674 | Gated driftAll=0.656 FARunk=0.661
[split 1 fold 2 fd=0.05] gateP=0.252 gateR=0.146 keep=0.231 | NoAdapt driftAll=0.625 FARunk=0.492 | Gated driftAll=0.671 FARunk=0.488
[split 1 fold 2 fd=0.10] gateP=0.243 gateR=0.19

In [9]:
def mean_std(vals):
    vals = np.array(vals, dtype=np.float64)
    return float(vals.mean()), float(vals.std(ddof=0))

variants = ["NoAdapt","UngatedAdapt","GatedAdapt","GatedConfAdapt","OracleGatePseudoAdapt","OracleLabelAdapt"]
metrics = ["stable_acc","drift_acc_rx","drift_acc_day","drift_acc_all","FRR_stable","FAR_unknown_accept","miss_drift_rx","miss_drift_day","miss_drift_all","auc_unknown_eval"]
base_keys = [
    "tau_id","tau_dom","tau_conf",
    "pseudo_acc_rx","pseudo_acc_day","pseudo_acc_all",
    "gate_precision","gate_recall","gate_keep_ratio",
    "conf_precision","conf_recall","conf_keep_ratio","conf_keep_given_gate"
]

summary = {"by_false_drift_target": {}}
fd_values = sorted(set(float(r["false_drift_target"]) for r in all_rows))

for fd in fd_values:
    rows_fd = [r for r in all_rows if float(r["false_drift_target"]) == fd]
    block = {}
    for key in base_keys:
        block[key] = dict(zip(["mean","std"], mean_std([r[key] for r in rows_fd])))
    block["variants"] = {}
    for v in variants:
        block["variants"][v] = {}
        for m in metrics:
            block["variants"][v][m] = dict(zip(["mean","std"], mean_std([r[v][m] for r in rows_fd])))
    summary["by_false_drift_target"][str(fd)] = block

with open(os.path.join(RUN_DIR, "summary_by_false_drift_target.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

for fd in fd_values:
    block = summary["by_false_drift_target"][str(fd)]
    print(f"\n================ FALSE_DRIFT_TARGET = {fd:.2f} ================")
    print("=== Drift pseudo-label accuracy (source model) ===")
    for key in ["pseudo_acc_rx","pseudo_acc_day","pseudo_acc_all"]:
        print(f"{key:20s}: {block[key]['mean']:.3f} ± {block[key]['std']:.3f}")

    print("\n=== Adaptation stream selection quality ===")
    for key in ["tau_id","tau_dom","tau_conf","gate_precision","gate_recall","gate_keep_ratio","conf_precision","conf_recall","conf_keep_ratio","conf_keep_given_gate"]:
        print(f"{key:20s}: {block[key]['mean']:.3f} ± {block[key]['std']:.3f}")

    for v in variants:
        print(f"\n=== {v} ===")
        for m in metrics:
            print(f"{m:20s}: {block['variants'][v][m]['mean']:.3f} ± {block['variants'][v][m]['std']:.3f}")

# save simple sweep plots
fds = np.array(fd_values, dtype=np.float32)

def collect(metric, variant):
    return np.array([summary["by_false_drift_target"][str(fd)]["variants"][variant][metric]["mean"] for fd in fd_values], dtype=np.float32)

plt.figure(figsize=(7,4))
for variant in ["NoAdapt","UngatedAdapt","GatedAdapt","GatedConfAdapt"]:
    plt.plot(fds, collect("drift_acc_all", variant), marker='o', label=variant)
plt.xlabel("FALSE_DRIFT_TARGET")
plt.ylabel("drift_acc_all")
plt.title("Drift accuracy vs false_drift_target")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "sweep_drift_acc_all.png"), dpi=160)
plt.close()

plt.figure(figsize=(7,4))
for variant in ["NoAdapt","UngatedAdapt","GatedAdapt","GatedConfAdapt"]:
    plt.plot(fds, collect("FAR_unknown_accept", variant), marker='o', label=variant)
plt.xlabel("FALSE_DRIFT_TARGET")
plt.ylabel("FAR_unknown_accept")
plt.title("Unknown FAR vs false_drift_target")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "sweep_far_unknown_accept.png"), dpi=160)
plt.close()

plt.figure(figsize=(7,4))
plt.plot(fds, np.array([summary["by_false_drift_target"][str(fd)]["gate_precision"]["mean"] for fd in fd_values]), marker='o', label='gate_precision')
plt.plot(fds, np.array([summary["by_false_drift_target"][str(fd)]["gate_recall"]["mean"] for fd in fd_values]), marker='o', label='gate_recall')
plt.plot(fds, np.array([summary["by_false_drift_target"][str(fd)]["gate_keep_ratio"]["mean"] for fd in fd_values]), marker='o', label='gate_keep_ratio')
plt.xlabel("FALSE_DRIFT_TARGET")
plt.ylabel("selection quality")
plt.title("Gate quality vs false_drift_target")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, "sweep_gate_quality.png"), dpi=160)
plt.close()

print("\nSaved grouped summary to:", os.path.join(RUN_DIR, "summary_by_false_drift_target.json"))
print("Saved plots:", ["sweep_drift_acc_all.png", "sweep_far_unknown_accept.png", "sweep_gate_quality.png"])


================ FALSE_DRIFT_TARGET = 0.01 ================
=== Drift pseudo-label accuracy (source model) ===
pseudo_acc_rx       : 0.611 ± 0.040
pseudo_acc_day      : 0.722 ± 0.070
pseudo_acc_all      : 0.639 ± 0.041

=== Adaptation stream selection quality ===
tau_id              : -1.307 ± 0.207
tau_dom             : 4953.841 ± 611.838
tau_conf            : 0.997 ± 0.003
gate_precision      : 0.489 ± 0.254
gate_recall         : 0.049 ± 0.013
gate_keep_ratio     : 0.058 ± 0.038
conf_precision      : 0.515 ± 0.251
conf_recall         : 0.048 ± 0.013
conf_keep_ratio     : 0.055 ± 0.039
conf_keep_given_gate: 0.934 ± 0.132

=== NoAdapt ===
stable_acc          : 0.997 ± 0.001
drift_acc_rx        : 0.610 ± 0.042
drift_acc_day       : 0.722 ± 0.074
drift_acc_all       : 0.666 ± 0.049
FRR_stable          : 0.058 ± 0.001
FAR_unknown_accept  : 0.280 ± 0.326
miss_drift_rx       : 0.247 ± 0.062
miss_drift_day      : 0.490 ± 0.069
miss_drift_all      : 0.369 ± 0.062
auc_unknown_eval    : 0.888 